## LANL Comprehensive, Multi-Source Cyber-Security Events — Gold ETL

**Reads from:** `workspace.silver_lanl.*`
**Writes to:** `workspace.gold_lanl.*`

### Gold Tables

| Gold Table | Description |
|---|---|
| `user_daily_auth_summary` | Per-user, per-day authentication behavior metrics |
| `computer_daily_network_profile` | Per-computer, per-day network + DNS activity |
| `hourly_security_dashboard` | Hourly aggregated threat-surface metrics for dashboarding |
| `user_process_baseline` | Per-user process behavior profile |
| `ml_user_day_features` | Wide ML feature table joining auth, network, and process signals per user-day |

All tables are written with `mode("overwrite")` for idempotent re-runs.

---

### Table of Contents

1. [Setup & Configuration](#1-setup--configuration)
2. [Gold Aggregation Tables](#2-gold-aggregation-tables) — per-user, per-computer, hourly rollups
3. [ML Feature Engineering](#3-ml-feature-engineering) — wide user-day feature table
4. [Validation](#4-validation) — row-count sanity checks

---

## 1. Setup & Configuration

Import aggregation functions, define catalog/schema constants, and create `workspace.gold_lanl`.

In [0]:
from pyspark.sql.functions import (
    col, sum, count, countDistinct, avg, max, min,
    when, lit, coalesce, current_timestamp
)

CATALOG = "workspace"
SILVER = "silver_lanl"
GOLD = "gold_lanl"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD}")
print(f"✓ Schema {CATALOG}.{GOLD} ready")

## 2. Gold Aggregation Tables

Business-ready aggregates built from silver tables:

| Table | Grain | Purpose |
|---|---|---|
| `user_daily_auth_summary` | user × day | Auth behavior metrics + compromise flag |
| `computer_daily_network_profile` | computer × day | Flow + DNS activity joined together |
| `hourly_security_dashboard` | hour | Threat-surface metrics for dashboarding |
| `user_process_baseline` | user × day | Process start/stop behavioral profile |

In [0]:
df_auth = spark.table(f"{CATALOG}.{SILVER}.auth_labeled")

df_user_daily_auth = (
    df_auth
    .groupBy("src_user", "src_domain", "time_day")
    .agg(
        count("*").alias("total_auth_events"),
        sum(col("is_failed").cast("int")).alias("failed_auth_events"),
        countDistinct("destination_computer").alias("distinct_dest_computers"),
        countDistinct("destination_user").alias("distinct_dest_users"),
        countDistinct("auth_type").alias("distinct_auth_types"),
        countDistinct("logon_type").alias("distinct_logon_types"),
        sum(col("is_compromised").cast("int")).alias("compromised_events"),
    )
    .withColumn(
        "fail_rate",
        (col("failed_auth_events") / col("total_auth_events"))
    )
)

df_user_daily_auth.write.mode("overwrite").saveAsTable(f"{CATALOG}.{GOLD}.user_daily_auth_summary")
print(f"✓ user_daily_auth_summary -> {CATALOG}.{GOLD}.user_daily_auth_summary")

In [0]:
# --- Sub-agg 1: network flows per computer per day ---
df_flows = spark.table(f"{CATALOG}.{SILVER}.flows")

df_flow_agg = (
    df_flows
    .groupBy("source_computer", "time_day")
    .agg(
        count("*").alias("total_flows"),
        sum("byte_count").alias("total_bytes"),
        sum("packet_count").alias("total_packets"),
        avg("duration").alias("avg_duration"),
        countDistinct("destination_computer").alias("distinct_dest_computers"),
        countDistinct("service_type").alias("distinct_services"),
        countDistinct("destination_port").alias("distinct_dest_ports"),
    )
)

# --- Sub-agg 2: DNS lookups per computer per day ---
df_dns = spark.table(f"{CATALOG}.{SILVER}.dns")

df_dns_agg = (
    df_dns
    .groupBy("source_computer", "time_day")
    .agg(
        count("*").alias("total_dns_lookups"),
        countDistinct("resolved_computer").alias("distinct_resolved"),
    )
)

# --- Full outer join on (source_computer, time_day) ---
df_network = (
    df_flow_agg
    .join(df_dns_agg, on=["source_computer", "time_day"], how="full_outer")
    .select(
        coalesce(df_flow_agg["source_computer"], df_dns_agg["source_computer"]).alias("source_computer"),
        coalesce(df_flow_agg["time_day"], df_dns_agg["time_day"]).alias("time_day"),
        coalesce("total_flows", lit(0)).alias("total_flows"),
        coalesce("total_bytes", lit(0)).alias("total_bytes"),
        coalesce("total_packets", lit(0)).alias("total_packets"),
        "avg_duration",
        coalesce("distinct_dest_computers", lit(0)).alias("distinct_dest_computers"),
        coalesce("distinct_services", lit(0)).alias("distinct_services"),
        coalesce("distinct_dest_ports", lit(0)).alias("distinct_dest_ports"),
        coalesce("total_dns_lookups", lit(0)).alias("total_dns_lookups"),
        coalesce("distinct_resolved", lit(0)).alias("distinct_resolved"),
    )
)

df_network.write.mode("overwrite").saveAsTable(f"{CATALOG}.{GOLD}.computer_daily_network_profile")
print(f"✓ computer_daily_network_profile -> {CATALOG}.{GOLD}.computer_daily_network_profile")

In [0]:
df_auth = spark.table(f"{CATALOG}.{SILVER}.auth_labeled")

df_hourly = (
    df_auth
    .groupBy("time_day", "time_hour")
    .agg(
        count("*").alias("total_auth"),
        sum(col("is_failed").cast("int")).alias("failed_auth"),
        sum(col("is_compromised").cast("int")).alias("compromised_auth"),
        countDistinct("src_user").alias("unique_users"),
        countDistinct("source_computer").alias("unique_src_computers"),
        countDistinct("destination_computer").alias("unique_dst_computers"),
    )
)

df_hourly.write.mode("overwrite").saveAsTable(f"{CATALOG}.{GOLD}.hourly_security_dashboard")
print(f"✓ hourly_security_dashboard -> {CATALOG}.{GOLD}.hourly_security_dashboard")

In [0]:
df_proc = spark.table(f"{CATALOG}.{SILVER}.proc")

df_proc_baseline = (
    df_proc
    .groupBy("proc_user", "proc_domain", "time_day")
    .agg(
        count("*").alias("total_process_events"),
        sum(col("is_start").cast("int")).alias("process_starts"),
        countDistinct("process_name").alias("distinct_processes"),
        countDistinct("computer").alias("distinct_computers"),
    )
    .withColumn("process_stops", col("total_process_events") - col("process_starts"))
)

df_proc_baseline.write.mode("overwrite").saveAsTable(f"{CATALOG}.{GOLD}.user_process_baseline")
print(f"✓ user_process_baseline -> {CATALOG}.{GOLD}.user_process_baseline")

## 3. ML Feature Engineering

Wide join of auth, process, and network signals into a single `ml_user_day_features` table — one row per (user, day) with all behavioral features and the `compromised_events` label.

In [0]:
# Wide ML feature table: joins auth, process, and network signals per user-day.
# This gives a single row per (user, day) with all behavioral features + label.

df_auth_gold = spark.table(f"{CATALOG}.{GOLD}.user_daily_auth_summary")
df_proc_gold = spark.table(f"{CATALOG}.{GOLD}.user_process_baseline")

# --- Join auth with process on (user, day) ---
df_ml = (
    df_auth_gold
    .join(
        df_proc_gold,
        on=[
            df_auth_gold["src_user"] == df_proc_gold["proc_user"],
            df_auth_gold["time_day"] == df_proc_gold["time_day"],
        ],
        how="left",
    )
    .select(
        df_auth_gold["src_user"].alias("user"),
        df_auth_gold["src_domain"].alias("domain"),
        df_auth_gold["time_day"],
        # Auth features
        "total_auth_events",
        "failed_auth_events",
        "fail_rate",
        "distinct_dest_computers",
        "distinct_dest_users",
        "distinct_auth_types",
        "distinct_logon_types",
        "compromised_events",
        # Process features (coalesce nulls for users with no process data)
        coalesce(col("total_process_events"), lit(0)).alias("total_process_events"),
        coalesce(col("process_starts"), lit(0)).alias("process_starts"),
        coalesce(col("process_stops"), lit(0)).alias("process_stops"),
        coalesce(col("distinct_processes"), lit(0)).alias("distinct_processes"),
        coalesce(col("distinct_computers"), lit(0)).alias("distinct_computers_proc"),
    )
)

# TODO: join with computer_daily_network_profile for network features.
#       Requires picking each user's most-active source_computer per day
#       (e.g., via a window function on auth events) as the join key.

# Label
df_ml = df_ml.withColumn(
    "is_compromised", (col("compromised_events") > 0)
)

df_ml.write.mode("overwrite").saveAsTable(f"{CATALOG}.{GOLD}.ml_user_day_features")
print(f"✓ ml_user_day_features -> {CATALOG}.{GOLD}.ml_user_day_features")

## 4. Validation

Row counts for all gold tables.

In [0]:
gold_tables = [
    "user_daily_auth_summary",
    "computer_daily_network_profile",
    "hourly_security_dashboard",
    "user_process_baseline",
    "ml_user_day_features",
]

counts = []
for t in gold_tables:
    n = spark.table(f"{CATALOG}.{GOLD}.{t}").count()
    counts.append((t, n))

df_counts = spark.createDataFrame(counts, ["table", "row_count"])
display(df_counts)